# Lab 1a: PySpark Bring-up, The "Obtain" Stage & Matrix Selection
**Course:** CIS 531/731 - Data Science and Analytics (Fall 2026)

**Objective:** Initialize a SparkSession, enforce schemas via `StructType`, handle malformed records, and lock in your project's Multidimensional Matrix architecture.

## Part 1: Environment Verification & PySpark Bring-Up

In [ ]:
# 1. Install PySpark (Required if running in Google Colab)
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

# 2. Initialize SparkSession (Local Mode)
spark = SparkSession.builder.appName("Lab1a_Obtain_Stage").master("local[*]").getOrCreate()

print(f"Spark Version: {spark.version}")
print(f"Spark UI Address: {spark.sparkContext.uiWebUrl}")
# HW1 Deliverable Note: Take a screenshot of the output above alongside your Spark Web UI tab.

### Track 1: VDB (Vector Database) - Batch Text Ingestion & Schema Enforcement
**Goal:** Enforce a strict schema and observe how Spark handles malformed records (`PERMISSIVE` vs `FAILFAST`).

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import tempfile
import os

# 1. Define Strict Schema
vdb_schema = StructType([
    StructField("doc_id", StringType(), False),
    StructField("author", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("text_chunk", StringType(), False)
])

# 2. Create Dummy JSON Data (Simulating a raw corpus dump with one bad row)
dummy_data = """{"doc_id": "doc_001", "author": "Author A", "year": 2024, "text_chunk": "The quick brown fox jumps over the lazy dog."}
{"doc_id": "doc_002", "author": "Author B", "year": 2025, "text_chunk": "A journey of a thousand miles begins with a single step."}
{"doc_id": "doc_003", "author": "Author C", "year": "NaN", "text_chunk": "Malformed year will trigger schema enforcement behavior."}"""

temp_json_path = os.path.join(tempfile.gettempdir(), "vdb_corpus.json")
with open(temp_json_path, "w") as f:
    f.write(dummy_data)

# 3. Execute Obtain Phase (Read Action)
# NOTE: We are using mode='PERMISSIVE' here so it doesn't crash the notebook, 
# but in production, you should use a Dead Letter Queue or mode='FAILFAST'.
vdb_df = spark.read.schema(vdb_schema).json(temp_json_path, mode="PERMISSIVE")

print("--- Schema Verification ---")
vdb_df.printSchema()

print("--- Ingested Data (Notice doc_003's year is null) ---")
vdb_df.show(truncate=False)

### Track 2: ASR (Live Audio Stream) - Structured Streaming Setup
**Goal:** Establish a streaming read context that bounds infinite data by capturing arriving chunks.

In [ ]:
import time

# 1. Setup a dummy stream directory to catch incoming buffers
stream_dir = os.path.join(tempfile.gettempdir(), "audio_stream_chunks")
os.makedirs(stream_dir, exist_ok=True)

# 2. Define schema for stream metadata (simulating audio buffer chunks)
asr_schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("chunk_id", StringType(), True),
    StructField("byte_size", IntegerType(), True)
])

# 3. Initialize Structured Streaming (ReadStream)
asr_stream = spark.readStream.schema(asr_schema).json(stream_dir)

# 4. WriteStream to memory sink for testing (Simulating passing data to Whisper API)
query = asr_stream.writeStream.format("memory").queryName("live_audio_buffer").start()

# 5. Simulate live data arriving over the network
with open(os.path.join(stream_dir, "chunk1.json"), "w") as f:
    f.write('{"timestamp": "10:00:01", "chunk_id": "chk_001", "byte_size": 1024}')

# Wait for the micro-batch to process the dropped file
time.sleep(3)

# 6. Check the in-memory sink to prove the Obtain phase captured the stream
print("--- Live Stream Buffer Contents ---")
spark.sql("SELECT * FROM live_audio_buffer").show()

# Clean up the streaming query
query.stop()

## Part 2: Term Project Architecture & Matrix Selection (HW1 Template)
**Instructions:** Double-click this cell to edit the Markdown. Fill in your team's architectural commitments based on the Multidimensional Matrix from Lecture 4. You can export this notebook as a PDF or copy this text into a separate document for your HW1 Canvas submission.

**1. Team Roster:**
- Member 1: [Name]
- Member 2: [Name]
- Member 3: [Name]

**2. Track Selection:**
- [ ] Track 1 (ASR Live Stream)
- [ ] Track 2 (VDB Text Corpora)

**3. Category Selection (Select at least 1):**
- [ ] Category 1: Recommendation, Info Retrieval, Prescriptive, Automation
- [ ] Category 2: Descriptive Analytics
- [ ] Category 3: Predictive Analytics, Generative Text for QA

**4. Task Matrix Allocation (Max 1 Task per member):**
- [Name 1]: [Claimed Task - e.g., Classification (Baseline Model)]
- [Name 2]: [Claimed Task - e.g., Classification (Alternative LLM)]
- [Name 3]: [Claimed Task - e.g., Descriptive Summarization Dashboard]

**5. Problem Statement (3-4 sentences):**
[Describe what your pipeline will actually do, combining your Track, Category, and Task. E.g., 'We will ingest a live gaming audio stream (ASR Track) to build a toxicity classification dashboard (Category 2). Student A will build the baseline classifier, Student B will fine-tune an alternative model, and Student C will build the descriptive visualization dashboard.']